[< Back to Main README](../README.md) | [Demo README](./README.md)

# Semantic Tool Selection: Token Efficiency and Accuracy Analysis

Based on: [Internal Representations as Indicators of Hallucinations in Agent Tool Selection](https://arxiv.org/pdf/2601.05214)

## What This Demo Measures

Three approaches run over the same 24 queries and are scored on two metrics.

| Approach | Tools sent per query |
|---|---|
| **Traditional** | all 29 |
| **Semantic** | top 3, selected by FAISS |
| **Semantic + Memory** | top 3, plus conversation history bounded to 3 turns |

1. **Token consumption**: what does each query cost?
2. **Tool selection accuracy**: does the agent call the tool the query asks for?

### The cost being attacked

Every tool schema is sent on every call. Traditional measures roughly 6,500 tokens per
query here, counting the tool schemas plus the system prompt, the user turn, tool
results, and model output.

### Read the accuracy result carefully

The headline result of this notebook is the token reduction, which is large and
reproducible. Accuracy is measured because it is what a cost optimization is most
likely to damage, not because filtering is expected to improve it.

Across runs so far the accuracy difference between Traditional and Semantic has
landed within a single query out of 24. That is too small a sample to call a
difference in either direction. The cells below print whatever they measure,
including a decrease, and they should be left that way. Do not adjust the query set
until the number flatters the technique. Reporting a measured result honestly,
including the part that is inconvenient, is the same discipline this workshop asks
you to apply to model output.

### The Pipeline

```
User Query → FAISS Search → Top 3 Tools → Agent → Tool Call
```


## ⚠️ Important: Execution Order

**This notebook must be executed sequentially from top to bottom.**

To run correctly:
1. Click **"Run All"** in the Jupyter menu, OR
2. Execute each cell in order using **Shift+Enter**

**Do NOT skip cells or run cells out of order** — each test depends on previous cells to define variables (`trad_results`, `sem_results`, `mem_results`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

## Configure AWS Credentials

This demo uses Amazon Bedrock (default model provider for Strands Agents). Ensure your AWS credentials are configured.

To use a different provider, see the [Model Providers documentation](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/).

In [ ]:
# Ensure AWS region is set (required for Bedrock in Workshop Studio)
import os
if not os.environ.get("AWS_DEFAULT_REGION") and not os.environ.get("AWS_REGION"):
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

import os
# Verify AWS credentials are available
import boto3
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")  # Show last 4 digits only
print(f"Region: {boto3.session.Session().region_name}")
print("\u2705 AWS credentials configured")

# This demo is Bedrock-only and needs no other provider credentials.
# To swap providers, see:
#   https://strandsagents.com/docs/user-guide/concepts/model-providers/


## Setup

In [ ]:
from typing import Dict, List, Tuple

from strands import Agent
# Model configuration — Amazon Bedrock (default, no extra import needed)
# Strands Agents uses Bedrock by default when no model is specified.
#
# To use a specific Bedrock model:
#   MODEL = "us.anthropic.claude-sonnet-5"
#   agent = Agent(tools=..., model=MODEL)
#
# See all providers: https://strandsagents.com/docs/user-guide/concepts/model-providers/
from enhanced_tools import ALL_TOOLS
from registry import (
    build_index,
    search_tools,
    swap_tools,
    trim_history,
    usage_delta,
    usage_snapshot,
)

# Conversation turns the memory agent retains. Unbounded history makes token
# cost grow quadratically across a 24-query run and lose to the baseline.
MEMORY_MAX_TURNS = 3

print(f"✅ Loaded {len(ALL_TOOLS)} tools")
print(f"✅ Memory agent history bounded to {MEMORY_MAX_TURNS} turns")

## Build Semantic Index

In [ ]:
build_index(ALL_TOOLS)
print("✅ FAISS index built")

## Test Queries

Diverse queries testing different tool categories:

In [ ]:
TESTS = [
    # Hotel queries - semantic filtering helps agent focus
    ("Search hotels in Barcelona", "search_hotels"),
    ("Find real hotels in France", "search_real_hotels"),
    ("Show me top rated hotels worldwide", "get_top_hotels"),
    ("What's the price for Grand Hotel Paris?", "get_hotel_pricing"),
    ("What amenities does the Hilton have?", "get_hotel_details"),
    ("Read reviews for Marriott Downtown", "search_hotel_reviews"),
    ("Is the Sheraton available tomorrow?", "check_hotel_availability"),
    ("Check availability March 15 to March 18", "check_hotel_availability_dates"),
    ("Compare hotel prices in Lisbon for next week", "compare_hotel_prices"),
    
    # Flight queries - similar tool names cause confusion with all 29 tools
    ("Find flights from NYC to Tokyo", "search_flights"),
    ("How much do flights to Paris cost?", "search_flight_prices"),
    ("Show me flight details for AA123", "get_flight_details"),
    ("Is flight BA456 on time?", "get_flight_status"),
    ("How many seats left on United 789?", "check_flight_availability"),
    
    # Booking workflow - semantic approach surfaces right tool sequence
    ("Book AnyCompany Hotel for John Smith", "book_hotel"),
    ("Book flight AA123 for Jane Doe", "book_flight"),
    ("Process payment of $500", "process_payment"),
    ("Confirm my booking BK-12345", "confirm_booking"),
    ("Cancel reservation BK-67890", "cancel_booking"),
    
    # Travel utilities - clear intent, semantic finds exact match
    ("Convert 500 USD to EUR", "get_currency_exchange"),
    ("Do I need a visa for Spain from USA?", "get_travel_documents"),
    ("What's the weather in Tokyo?", "get_weather"),
    ("Show me weather forecast for London", "get_weather_forecast"),
    ("Any weather alerts in Miami?", "get_weather_alerts"),
]

print(f"📋 Test suite: {len(TESTS)} realistic queries")
print(f"📊 Tool pool: {len(ALL_TOOLS)} tools")
print(f"🎯 Goal: measure the token/accuracy tradeoff, not assume filtering improves accuracy")

## Helper Functions

In [ ]:
def run_and_capture_with_tokens(agent, query: str, verbose: bool = False) -> Tuple[List[str], Dict]:
    """Run agent and capture tool calls + this call's token usage.

    accumulated_usage is the agent's lifetime counter, so it is differenced
    against a pre-call snapshot. Reading it directly would make the reused
    memory agent report a running total and sum to a triangular number.
    """
    before = usage_snapshot(agent)
    result = agent(query)
    
    # Extract tool calls from result.metrics.tool_metrics
    tools = list(result.metrics.tool_metrics.keys()) if result.metrics else []
    
    # Use Strands native token counting, per call rather than cumulative
    tokens = {'input': 0, 'output': 0, 'total': 0, 'estimated': False}
    if result.metrics:
        used = usage_delta(agent, before)
        tokens['input'] = used['inputTokens']
        tokens['output'] = used['outputTokens']
        tokens['total'] = used['totalTokens']
    
    # Optionally print verbose output (agent response)
    if verbose:
        print(f"    Agent response: {result.message['content'][0]['text'][:200]}...")
    
    return tools, tokens

## Test 1: Traditional Approach (All 29 Tools)

Agent receives all tools on every query.

In [ ]:
PROMPT = "You are a travel assistant. Use the correct tool to answer questions."

print("="*80)
print(f"TEST 1: TRADITIONAL - {len(ALL_TOOLS)} tools every query")
print("="*80)
print()

trad_results = []
trad_correct = 0
trad_total_tokens = 0

for i, (query, expected) in enumerate(TESTS, 1):
    agent = Agent(tools=ALL_TOOLS, system_prompt=PROMPT)
    tools, tokens = run_and_capture_with_tokens(agent, query, verbose=False)
    
    ok = expected in tools
    trad_correct += ok
    trad_total_tokens += tokens['total']
    
    trad_results.append({
        'query': query,
        'expected': expected,
        'actual': tools,
        'correct': ok,
        'tokens': tokens
    })

print()
print(f"📊 Traditional Results:")
print(f"   Accuracy: {trad_correct}/{len(TESTS)} ({100*trad_correct/len(TESTS):.1f}%)")
print(f"   Total tokens: {trad_total_tokens:,}")
print(f"   Avg tokens/query: {trad_total_tokens/len(TESTS):.0f}")

In [ ]:
# Compact summary table for Test 1
print("\n" + "="*80)
print("TEST 1 SUMMARY TABLE")
print("="*80)
print(f"\n{'#':>3} {'Query':<45} {'Expected':<20} {'Called':<20} {'Result':>6} {'Tokens':>7}")
print("-"*80)

for i, r in enumerate(trad_results, 1):
    status = '✅' if r['correct'] else '❌'
    query_short = r['query'][:44]
    expected = r['expected'][:19]
    actual = ', '.join(r['actual'][:2])[:19] if r['actual'] else 'NO TOOL'
    tokens = r['tokens']['total']
    print(f"{i:3} {query_short:<45} {expected:<20} {actual:<20} {status:>6} {tokens:7,}")

print("-"*80)
print(f"Total: {trad_correct}/{len(TESTS)} correct ({100*trad_correct/len(TESTS):.1f}%), {trad_total_tokens:,} tokens")

## Test 2: Semantic Approach (Top-3 Tools)

FAISS selects top-3 relevant tools per query.

In [ ]:
print("="*80)
print("TEST 2: SEMANTIC - Top-3 tools per query")
print("="*80)
print()

sem_results = []
sem_correct = 0
sem_total_tokens = 0

for i, (query, expected) in enumerate(TESTS, 1):
    selected = search_tools(query, top_k=3)
    selected_names = [t.__name__ for t in selected]
    
    agent = Agent(tools=selected, system_prompt=PROMPT)
    tools, tokens = run_and_capture_with_tokens(agent, query, verbose=False)
    
    ok = expected in tools
    sem_correct += ok
    sem_total_tokens += tokens['total']
    
    sem_results.append({
        'query': query,
        'expected': expected,
        'selected': selected_names,
        'actual': tools,
        'correct': ok,
        'tokens': tokens
    })

print()
print(f"📊 Semantic Results:")
print(f"   Accuracy: {sem_correct}/{len(TESTS)} ({100*sem_correct/len(TESTS):.1f}%)")
print(f"   Total tokens: {sem_total_tokens:,}")
print(f"   Avg tokens/query: {sem_total_tokens/len(TESTS):.0f}")

In [ ]:
# Compact summary table for Test 2
print("\n" + "="*80)
print("TEST 2 SUMMARY TABLE")
print("="*80)
print(f"\n{'#':>3} {'Query':<45} {'Expected':<20} {'Called':<20} {'Result':>6} {'Tokens':>7}")
print("-"*80)

for i, r in enumerate(sem_results, 1):
    status = '✅' if r['correct'] else '❌'
    query_short = r['query'][:44]
    expected = r['expected'][:19]
    actual = ', '.join(r['actual'][:2])[:19] if r['actual'] else 'NO TOOL'
    tokens = r['tokens']['total']
    print(f"{i:3} {query_short:<45} {expected:<20} {actual:<20} {status:>6} {tokens:7,}")
    # Show available tools for failures
    if not r['correct']:
        available = ', '.join(r['selected'])
        print(f"    └─ Available tools: [{available}]")
        if r['expected'] not in r['selected']:
            print(f"       ⚠️  Correct tool NOT in top-3 (FAISS filtering issue)")

print("-"*80)
print(f"Total: {sem_correct}/{len(TESTS)} correct ({100*sem_correct/len(TESTS):.1f}%), {sem_total_tokens:,} tokens")

## Test 3: Semantic + Memory (Single Agent)

Same agent across all queries, tools swapped dynamically.

Conversation history is **bounded** to the last `MEMORY_MAX_TURNS` turns. Without a
bound the full transcript is resent on every call, so cost grows quadratically with
turn count and this variant becomes far more expensive than the traditional baseline.
Bounding is what makes memory viable — watch the message count stay flat below.

In [ ]:
print("="*80)
print("TEST 3: SEMANTIC + MEMORY - Single agent, dynamic tool swapping,")
print(f"        conversation history bounded to {MEMORY_MAX_TURNS} turns")
print("="*80)
print()

initial_tools = search_tools(TESTS[0][0], top_k=3)
memory_agent = Agent(tools=initial_tools, system_prompt=PROMPT)

mem_results = []
mem_correct = 0
mem_total_tokens = 0

for i, (query, expected) in enumerate(TESTS, 1):
    selected = search_tools(query, top_k=3)
    selected_names = [t.__name__ for t in selected]
    swap_tools(memory_agent, selected)
    trim_history(memory_agent, max_turns=MEMORY_MAX_TURNS)
    
    tools, tokens = run_and_capture_with_tokens(memory_agent, query, verbose=False)
    
    ok = expected in tools
    mem_correct += ok
    mem_total_tokens += tokens['total']
    
    mem_results.append({
        'query': query,
        'expected': expected,
        'selected': selected_names,
        'actual': tools,
        'correct': ok,
        'tokens': tokens,
        'messages': len(memory_agent.messages)
    })

print()
print(f"📊 Semantic+Memory Results:")
print(f"   Accuracy: {mem_correct}/{len(TESTS)} ({100*mem_correct/len(TESTS):.1f}%)")
print(f"   Total tokens: {mem_total_tokens:,}")
print(f"   Avg tokens/query: {mem_total_tokens/len(TESTS):.0f}")
print(f"   Final conversation: {len(memory_agent.messages)} messages (bounded to {MEMORY_MAX_TURNS} turns)")
print(f"   Peak conversation:  {max(r['messages'] for r in mem_results)} messages")

In [ ]:
# Compact summary table for Test 3
print("\n" + "="*80)
print("TEST 3 SUMMARY TABLE")
print("="*80)
print(f"\n{'#':>3} {'Query':<45} {'Expected':<20} {'Called':<20} {'Result':>6} {'Tokens':>7}")
print("-"*80)

for i, r in enumerate(mem_results, 1):
    status = '✅' if r['correct'] else '❌'
    query_short = r['query'][:44]
    expected = r['expected'][:19]
    actual = ', '.join(r['actual'][:2])[:19] if r['actual'] else 'NO TOOL'
    tokens = r['tokens']['total']
    print(f"{i:3} {query_short:<45} {expected:<20} {actual:<20} {status:>6} {tokens:7,}")
    # Show available tools for failures
    if not r['correct']:
        available = ', '.join(r['selected'])
        print(f"    └─ Available tools: [{available}]")

print("-"*80)
print(f"Total: {mem_correct}/{len(TESTS)} correct ({100*mem_correct/len(TESTS):.1f}%), {mem_total_tokens:,} tokens, {len(memory_agent.messages)} messages")

## Comparative Analysis

In [ ]:
print("="*80)
print("COMPARATIVE ANALYSIS")
print("="*80)

print(f"\n📊 Accuracy Comparison:")
print(f"   Traditional:      {trad_correct}/{len(TESTS)} ({100*trad_correct/len(TESTS):.1f}%)")
print(f"   Semantic:         {sem_correct}/{len(TESTS)} ({100*sem_correct/len(TESTS):.1f}%)")
print(f"   Semantic+Memory:  {mem_correct}/{len(TESTS)} ({100*mem_correct/len(TESTS):.1f}%)")

# Print the measured accuracy gap and its sample size together. A gap of one or
# two queries out of 24 is not a result in either direction, and saying so here
# is what stops the token reduction from being read as an accuracy claim too.
gap = sem_correct - trad_correct
if gap == 0:
    print(f"   ➡️  Same accuracy as Traditional")
else:
    direction = "above" if gap > 0 else "below"
    print(f"   Semantic scored {abs(gap)} quer{'y' if abs(gap) == 1 else 'ies'} "
          f"{direction} Traditional, out of {len(TESTS)}.")
    if abs(gap) <= 2:
        print(f"   ⚖️  Within noise at n={len(TESTS)}. Not evidence that filtering")
        print(f"      helps or hurts accuracy. The token reduction below is the")
        print(f"      result this demo establishes.")

print(f"\n💰 Token Consumption:")
print(f"   Traditional:      {trad_total_tokens:,} tokens ({trad_total_tokens/len(TESTS):.0f} avg)")
print(f"   Semantic:         {sem_total_tokens:,} tokens ({sem_total_tokens/len(TESTS):.0f} avg)")
print(f"   Semantic+Memory:  {mem_total_tokens:,} tokens ({mem_total_tokens/len(TESTS):.0f} avg)")

if trad_total_tokens > 0:
    sem_savings = trad_total_tokens - sem_total_tokens
    mem_savings = trad_total_tokens - mem_total_tokens
    def _verdict(saved):
        pct = 100 * saved / trad_total_tokens
        word = "reduction" if saved > 0 else "INCREASE"
        return f"{abs(saved):,} tokens ({abs(pct):.1f}% {word})"

    print(f"\n💡 Token Savings (measured this run, not a cited figure):")
    print(f"   Semantic vs Traditional:  {_verdict(sem_savings)}")
    print(f"   Memory vs Traditional:    {_verdict(mem_savings)}")
    if mem_savings <= 0:
        print(f"   ⚠️  Memory loses to the baseline even with bounded history —")
        print(f"      conversation context outweighs the tool-schema saving here.")
    
    if mem_total_tokens > sem_total_tokens:
        overhead = mem_total_tokens - sem_total_tokens
        print(f"   Memory overhead:          +{overhead:,} tokens (conversation history)")

## Per-Query Token Breakdown

In [ ]:
print("="*80)
print("PER-QUERY TOKEN BREAKDOWN")
print("="*80)

print(f"\n{'Query':<50} {'Trad':>8} {'Sem':>8} {'Mem':>8} {'Saved':>8}")
print("-"*80)

for i in range(len(TESTS)):
    query = TESTS[i][0][:49]
    trad_tok = trad_results[i]['tokens']['total']
    sem_tok = sem_results[i]['tokens']['total']
    mem_tok = mem_results[i]['tokens']['total']
    saved = trad_tok - sem_tok
    
    print(f"{query:<50} {trad_tok:8} {sem_tok:8} {mem_tok:8} {saved:8}")

## Error Analysis

In [ ]:
print("="*80)
print("ERROR ANALYSIS")
print("="*80)

print(f"\n🔍 Traditional Errors:")
trad_errors = [r for r in trad_results if not r['correct']]
if trad_errors:
    for r in trad_errors:
        print(f"   ❌ '{r['query'][:60]}'")
        print(f"      Expected: {r['expected']}, Got: {r['actual']}")
else:
    print("   ✅ No errors")

print(f"\n🔍 Semantic Errors:")
sem_errors = [r for r in sem_results if not r['correct']]
if sem_errors:
    for r in sem_errors:
        print(f"   ❌ '{r['query'][:60]}'")
        print(f"      Expected: {r['expected']}, Got: {r['actual']}")
        print(f"      Available: {r['selected']}")
        if r['expected'] not in r['selected']:
            print(f"      ⚠️  Correct tool NOT in top-3 (FAISS filtering issue)")
else:
    print("   ✅ No errors")

print(f"\n🔍 Semantic+Memory Errors:")
mem_errors = [r for r in mem_results if not r['correct']]
if mem_errors:
    for r in mem_errors:
        print(f"   ❌ '{r['query'][:60]}'")
        print(f"      Expected: {r['expected']}, Got: {r['actual']}")
        print(f"      Available: {r['selected']}")
else:
    print("   ✅ No errors")

---
## Summary

### Key Findings

| Approach | Tools/Query | Accuracy | Avg Tokens | Token Savings |
|----------|-------------|----------|------------|---------------|
| Traditional | 29 tools | X% | ~Y tokens | Baseline |
| Semantic | 3 tools | X% | ~Y tokens | Z% |
| Semantic+Memory | 3 tools | X% | ~Y tokens | Z% |

*(Actual values populated after running the cells above)*

### Implementation Notes

Strands provides native dynamic tool swapping. No agent recreation, no conversation loss:

```python
# One agent, dynamic tools
agent = Agent(tools=initial_tools, model=MODEL)

for query in queries:
    selected = search_tools(query, top_k=3)   # FAISS → top 3
    swap_tools(agent, selected)                # swap registry, keep memory
    agent(query)                               # conversation history preserved
```

`swap_tools()` works because Strands calls `tool_registry.get_all_tools_config()` at each event loop cycle. Tool changes are picked up automatically without restarting the agent.

→ [Strands Tool Registry](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/)

### What Semantic Tool Selection Buys You

1. **A large, reproducible token reduction.** Sending 3 tool schemas instead of 29 is a
   constant saving on every call. This is the result the demo establishes.
2. **Accuracy is a tradeoff to watch, not a win to claim.** Filtering can only help if
   the correct tool survives the top-3 cut. When it does not, the agent cannot recover,
   and the Error Analysis cell above labels exactly those cases. Measure accuracy on
   your own tool set and query mix before adopting this; do not assume the reduction
   comes for free.
3. **Production ready.** `swap_tools()` preserves conversation memory across queries.

### Reading the numbers above

Everything printed by this notebook is LLM output and moves between runs. Treat the
token reduction as a range rather than a fixed figure, and treat any accuracy gap of
one or two queries out of 24 as noise until a larger evaluation says otherwise.

---

## References

### Research
- [Internal Representations as Indicators of Hallucinations in Agent Tool Selection](https://arxiv.org/pdf/2601.05214)

### Strands Agents
- [Strands Tool Registry](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/) — Dynamic tool management
- [Creating Custom Tools](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/) — `@tool` decorator
- [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/) — Swap to Amazon Bedrock, Anthropic, Ollama
- [Strands Agents Documentation](https://strandsagents.com) — Full framework docs

### Code
- [Code Repository](https://github.com/aws-samples/sample-stop-ai-agent-hallucinations-workshop)
